In [ ]:
#Install necessary packages
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

In [33]:
#Setting up environment variables
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, CodeInterpreterTool, CodeInterpreterToolAuto

load_dotenv()

foundry_project_endpoint =os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")


In [34]:
#Setting up the project client
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

In [35]:
#Creating the OpenAI Client
openai_client = project_client.get_openai_client()

In [36]:
#upload the excel file for the code interpreter tool to access
file = openai_client.files.create(purpose="assistants", file=open("./travelExpense.xlsx", "rb"))
print(f"File uploaded with ID: {file.id}")

File uploaded with ID: assistant-TUjHuQ28YNhUNWHJAtaVn5


In [45]:
#Creating Agent with the code interpreter tool
agent_name = "code-interpreter-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(   
        model=model_deployment_name,
        instructions="You are a helpful assistant that helps the user analyze their travel expenses. You have access to a file called travelExpense.xlsx which contains all the data about the user's travel expenses. Use this data to answer the user's queries about their travel expenses.",
        tools=[
            CodeInterpreterTool(
                container = CodeInterpreterToolAuto(
                    file_ids=[file.id]
                )
            )
        ]
    ),
)
print(f"Agent created with name: {agent.id} {agent.name} and version: {agent.version}")

Agent created with name: code-interpreter-agent:1 code-interpreter-agent and version: 1


In [46]:
#Creating the conversation object for the agent to interact with
conversation = openai_client.conversations.create()
print(f"Conversation created with ID: {conversation.id}")

Conversation created with ID: conv_77e9e89ae683e73700RXqsSKOOxEBbb7kf2smSDFIInBXxIoNY


In [50]:
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={
        "agent": {
            "name": agent.name,
            "type":"agent_reference"
        }
    },
    input = "Could you please analyze my travel expenses and create a column chart with categories on the x-axis and amount spent on the y-axis?"
    
)
print(f"Agent response: {response.id}")

Agent response: resp_77e9e89ae683e737006a26726753188190959da5d9a38f86af


In [52]:
#Extracting file information from the response to display the chart
file_id = ""
filename=""
container_id = ""

#Get the last message which contains file citations

last_message = response.output[-1]

if last_message.type == "message":
    #get the last content item (contains file annotation)
    text_content = last_message.content[-1]
    if text_content.annotations:
        file_citation= text_content.annotations[-1]
        if file_citation.type == "container_file_citation":
            file_id = file_citation.file_id
            filename = file_citation.filename
            container_id = file_citation.container_id
            print(f"File ID: {file_id}, Filename: {filename}, Container ID: {container_id}")

#download the file using the file id
if file_id and filename:
    file_content = openai_client.containers.files.content.retrieve(file_id=file_id, container_id=container_id)
    with open(f"downloaded_{filename}", "wb") as f:
        f.write(file_content.read())
        print(f"File downloaded successfully as downloaded_{filename}")
else:
    print("No file citation found in the response.")


File ID: cfile_6a26728167f08190aae1948c6c6017bc, Filename: travel_expenses_by_category.png, Container ID: cntr_6a26726b75fc81909bd845841a42da79052ef74f8fbc83d6
File downloaded successfully as downloaded_travel_expenses_by_category.png
